# Tennis domain adaptation experiments

This notebook orchestrates the repository scripts for local Antigravity/Jupyter runs and Colab GPU runs. It does not duplicate model loading, generation, scoring, or training logic. Start with smoke-mode commands before enabling full runs.

## 1. Runtime setup

These cells detect the runtime, resolve the repository root from the current checkout, and move the kernel into the repo root. Google Drive mounting and cloning are optional Colab-only conveniences.

In [ ]:
from shutil import which
import subprocess

if which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("nvidia-smi not found; continuing without a visible NVIDIA GPU.")

In [ ]:
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount("/content/drive")
except Exception as exc:
    IN_COLAB = False
    print(f"Google Drive mount skipped: {exc}")

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Antigravity/local: leave empty and the notebook will find this checkout from cwd.
# Colab: either upload/clone the repo, or set this to your Drive checkout path.
PROJECT_ROOT_OVERRIDE = ""

# Optional. In Colab, set this if the repo is not already present.
GITHUB_REPO_URL = ""

def has_repo_markers(path):
    path = Path(path)
    return (
        (path / "pyproject.toml").exists()
        and (path / "scripts" / "tennis" / "evaluate_tennis.py").exists()
        and (path / "data" / "tennis" / "tennis_test.json").exists()
    )

def candidate_roots():
    seen = set()
    starts = [Path.cwd()]
    notebook_file = globals().get("__vsc_ipynb_file__") or globals().get("__file__")
    if notebook_file:
        starts.append(Path(notebook_file).expanduser().resolve().parent)
    for start in starts:
        start = Path(start).expanduser().resolve()
        for candidate in (start, *start.parents):
            if candidate not in seen:
                seen.add(candidate)
                yield candidate

def discover_project_root():
    if PROJECT_ROOT_OVERRIDE:
        override = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
        if has_repo_markers(override):
            return override
        raise FileNotFoundError(f"PROJECT_ROOT_OVERRIDE is not this repo checkout: {override}")

    for candidate in candidate_roots():
        if has_repo_markers(candidate):
            return candidate

    if IN_COLAB and GITHUB_REPO_URL:
        clone_target = Path("/content/tiser_temporal_reasoning_extension")
        if not clone_target.exists():
            subprocess.run(["git", "clone", GITHUB_REPO_URL, str(clone_target)], check=True)
        if has_repo_markers(clone_target):
            return clone_target

    raise FileNotFoundError(
        "Could not find the repo root. Open the notebook from inside the checkout "
        "or set PROJECT_ROOT_OVERRIDE to the absolute repo path."
    )

PROJECT_ROOT = discover_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.environ["PYTHONPATH"] = str(PROJECT_ROOT) + os.pathsep + os.environ.get("PYTHONPATH", "")

print(f"Project root: {PROJECT_ROOT}")
print(f"Working directory: {Path.cwd()}")

In [ ]:
%cd {PROJECT_ROOT}

## 2. Install dependencies

In [ ]:
import subprocess
import sys
from pathlib import Path

requirements = Path("requirements.txt")
if requirements.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(requirements)], check=True)
else:
    print("requirements.txt not found; skipping install.")

In [ ]:
import importlib.util
import subprocess
import sys

runtime_dependencies = {
    'accelerate': 'accelerate',
    'bitsandbytes': 'bitsandbytes',
    'datasets': 'datasets',
    'peft': 'peft',
    'torch': 'torch',
    'transformers': 'transformers',
    'trl': 'trl',
    'yaml': 'pyyaml',
}
missing = [package for module, package in runtime_dependencies.items() if importlib.util.find_spec(module) is None]
if missing:
    print('Installing missing runtime dependencies:', missing)
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
else:
    print('All checked runtime dependencies are importable.')

In [ ]:
import platform
import torch

print('Python:', platform.python_version())
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device:', torch.cuda.get_device_name(0))

## 3. Configuration flags

Full 7B evaluations and training are disabled by default. Enable only the stages you intend to run.

In [ ]:
RUN_SMOKE = True
RUN_BASE_QWEN = False
RUN_ORIGINAL_TISER = False
RUN_TENNIS_ONLY_EVAL = False
RUN_MIXED_REPLAY_EVAL = False
RUN_AGGREGATION = True
RUN_TRAINING = False

LIMIT = 5
BATCH_SIZE = 1
MAX_NEW_TOKENS = 256

## 4. Path configuration

In [ ]:
TENNIS_TEST = 'data/tennis/tennis_test.json'
TENNIS_TRAIN_TRACED = 'data/tennis/tennis_train_traced.json'

ORIGINAL_TISER_ADAPTER = 'model/tiser_qwen7b_full/adapter'
TENNIS_ONLY_ADAPTER = 'model/tennis_only_qwen7b/adapter'
MIXED_REPLAY_ADAPTER = 'model/mixed_tennis_tiser_replay_qwen7b/adapter'

CONFIG = 'config/config_tennis.yaml'
SMOKE_CONFIG = 'config/config_tennis_smoke.yaml'

RESULTS_DIR = 'results/tennis_domain_adaptation'

In [ ]:
import shlex
import subprocess
import sys
import textwrap

def normalize_command(command):
    lines = textwrap.dedent(command).strip().splitlines()
    normalized = []
    for line in lines:
        stripped = line.strip()
        if not stripped:
            continue
        if stripped.endswith("\\"):
            stripped = stripped[:-1].rstrip()
        normalized.append(stripped)
    return " ".join(normalized)

def run_shell(command, *, enabled=True):
    command = normalize_command(command)
    if not enabled:
        print("[skip] Stage disabled. Command retained for reproducibility:")
        print(command)
        return
    print("[run]")
    print(command)
    args = shlex.split(command, posix=True)
    if args and args[0] == "python":
        args[0] = sys.executable
    subprocess.run(args, check=True)

## 5. Preflight checks

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display

def preflight_row(label, path, *, kind='file', required=True, enabled=True):
    p = Path(path)
    exists = p.is_dir() if kind == 'dir' else p.exists()
    if exists:
        status = 'PASS'
        note = 'found'
    elif enabled and required:
        status = 'FAIL'
        note = 'missing and required for enabled stage'
    else:
        status = 'WARN'
        note = 'missing; stage is disabled or script is optional'
    return {'status': status, 'item': label, 'path': str(p), 'note': note}

checks = [
    preflight_row('tennis test file', TENNIS_TEST),
    preflight_row('tennis train traced file', TENNIS_TRAIN_TRACED, required=False, enabled=RUN_TRAINING),
    preflight_row('config', CONFIG),
    preflight_row('smoke config', SMOKE_CONFIG),
    preflight_row('original TISER adapter', ORIGINAL_TISER_ADAPTER, kind='dir', enabled=RUN_ORIGINAL_TISER),
    preflight_row('tennis-only adapter', TENNIS_ONLY_ADAPTER, kind='dir', enabled=RUN_TENNIS_ONLY_EVAL),
    preflight_row('mixed replay adapter', MIXED_REPLAY_ADAPTER, kind='dir', enabled=RUN_MIXED_REPLAY_EVAL),
    preflight_row('evaluate_tennis.py', 'scripts/tennis/evaluate_tennis.py'),
    preflight_row('compare_adapters.py', 'scripts/tennis/compare_adapters.py'),
    preflight_row('run_experiment_plan.py', 'scripts/tennis/run_experiment_plan.py', required=False, enabled=False),
    preflight_row('aggregate_tennis_results.py', 'scripts/tennis/aggregate_tennis_results.py', required=False, enabled=False),
    preflight_row('train_tennis.py', 'scripts/tennis/train_tennis.py', required=False, enabled=RUN_TRAINING),
]

header = '| Status | Item | Path | Note |\n|---|---|---|---|'
body = '\n'.join(f"| {row['status']} | {row['item']} | `{row['path']}` | {row['note']} |" for row in checks)
display(Markdown(header + '\n' + body))

failures = [row for row in checks if row['status'] == 'FAIL']
if failures:
    raise FileNotFoundError('Preflight failed for: ' + ', '.join(row['item'] for row in failures))

## 6. Smoke evaluation

In [ ]:
run_shell(f'''
python scripts/tennis/evaluate_tennis.py \
  --config "{CONFIG}" \
  --test-file "{TENNIS_TEST}" \
  --condition base_qwen_smoke \
  --no-adapter \
  --limit {LIMIT} \
  --batch-size 1 \
  --max-new-tokens 256 \
  --output-dir "{RESULTS_DIR}/scored/base_qwen_smoke"
''', enabled=RUN_SMOKE)

## 7. Full baseline evaluations

In [ ]:
run_shell(f'''
python scripts/tennis/evaluate_tennis.py \
  --config "{CONFIG}" \
  --test-file "{TENNIS_TEST}" \
  --condition base_qwen \
  --no-adapter \
  --batch-size {BATCH_SIZE} \
  --max-new-tokens {MAX_NEW_TOKENS} \
  --output-dir "{RESULTS_DIR}/scored/base_qwen"
''', enabled=RUN_BASE_QWEN)

In [ ]:
run_shell(f'''
python scripts/tennis/evaluate_tennis.py \
  --config "{CONFIG}" \
  --test-file "{TENNIS_TEST}" \
  --adapter-dir "{ORIGINAL_TISER_ADAPTER}" \
  --condition original_tiser \
  --batch-size {BATCH_SIZE} \
  --max-new-tokens {MAX_NEW_TOKENS} \
  --output-dir "{RESULTS_DIR}/scored/original_tiser"
''', enabled=RUN_ORIGINAL_TISER)

## 8. Tennis adapter evaluations

In [ ]:
run_shell(f'''
python scripts/tennis/evaluate_tennis.py \
  --config "{CONFIG}" \
  --test-file "{TENNIS_TEST}" \
  --adapter-dir "{TENNIS_ONLY_ADAPTER}" \
  --condition tennis_only \
  --batch-size {BATCH_SIZE} \
  --max-new-tokens {MAX_NEW_TOKENS} \
  --output-dir "{RESULTS_DIR}/scored/tennis_only"
''', enabled=RUN_TENNIS_ONLY_EVAL)

In [ ]:
run_shell(f'''
python scripts/tennis/evaluate_tennis.py \
  --config "{CONFIG}" \
  --test-file "{TENNIS_TEST}" \
  --adapter-dir "{MIXED_REPLAY_ADAPTER}" \
  --condition mixed_tennis_tiser_replay \
  --batch-size {BATCH_SIZE} \
  --max-new-tokens {MAX_NEW_TOKENS} \
  --output-dir "{RESULTS_DIR}/scored/mixed_tennis_tiser_replay"
''', enabled=RUN_MIXED_REPLAY_EVAL)

## 9. Experiment plan runner

This writes `results/tennis_domain_adaptation/comparisons/run_tennis_experiments.sh` in dry-run mode when the plan script exists. It does not execute the generated full experiment script.

In [ ]:
if Path('scripts/tennis/run_experiment_plan.py').exists():
    run_shell(f'''
python scripts/tennis/run_experiment_plan.py \
  --config "{CONFIG}" \
  --tennis-test "{TENNIS_TEST}" \
  --original-tiser-adapter "{ORIGINAL_TISER_ADAPTER}" \
  --tennis-adapter "{TENNIS_ONLY_ADAPTER}" \
  --mixed-adapter "{MIXED_REPLAY_ADAPTER}" \
  --limit {LIMIT}
''', enabled=True)
else:
    print('scripts/tennis/run_experiment_plan.py not found; skipping dry-run plan generation.')

In [ ]:
run_shell(f'''
python scripts/tennis/train_tennis.py \
  --config "{CONFIG}" \
  --train-file "{TENNIS_TRAIN_TRACED}" \
  --run-name tennis_only_qwen7b
''', enabled=RUN_TRAINING)

## 10. Aggregation

In [ ]:
run_shell(f'''
python scripts/tennis/compare_adapters.py \
  --results-dir "{RESULTS_DIR}"
''', enabled=RUN_AGGREGATION)

if Path('scripts/tennis/aggregate_tennis_results.py').exists():
    run_shell(f'''
python scripts/tennis/aggregate_tennis_results.py \
  --results-dir "{RESULTS_DIR}"
''', enabled=RUN_AGGREGATION)
else:
    print('scripts/tennis/aggregate_tennis_results.py not found; skipping optional aggregation.')

## 11. Display result summaries

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display

summary_files = [
    Path(RESULTS_DIR) / 'comparisons/adapter_comparison.md',
    Path(RESULTS_DIR) / 'comparisons/final_results_table.md',
    Path(RESULTS_DIR) / 'comparisons/category_analysis.md',
    Path(RESULTS_DIR) / 'comparisons/forgetting_analysis.md',
]

for path in summary_files:
    if path.exists():
        display(Markdown(f'### {path.as_posix()}'))
        display(Markdown(path.read_text(encoding='utf-8')))
    else:
        print(f'[missing] {path.as_posix()}')

## 12. Save artifacts

In [ ]:
from pathlib import Path
import shutil

COPY_ZIP_TO_DRIVE = False
DRIVE_ZIP_TARGET = Path('/content/drive/MyDrive/tennis_domain_adaptation_results.zip')

results_path = Path(RESULTS_DIR)
if not results_path.exists():
    print(f'Results directory does not exist yet: {results_path}')
else:
    archive_path = shutil.make_archive(
        'tennis_domain_adaptation_results',
        'zip',
        root_dir=results_path.parent,
        base_dir=results_path.name,
    )
    print(f'Wrote {archive_path}')
    if COPY_ZIP_TO_DRIVE:
        DRIVE_ZIP_TARGET.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(archive_path, DRIVE_ZIP_TARGET)
        print(f'Copied to {DRIVE_ZIP_TARGET}')

## 13. Safety notes

- Do not run full 7B evaluations unless GPU memory is sufficient.
- Start with `--limit 5`.
- The smoke config using Qwen2.5-0.5B is only for base-model smoke checks, not for 7B adapters.
- Adapter model must match Qwen/Qwen2.5-7B-Instruct.
- Training should only be run after `tennis_train_traced.json` has validated TISER outputs.
- Keep checkpoints and large generated outputs out of git; final experiment summaries should live under `results/tennis_domain_adaptation/`.